**You need to INCREMENTALLY STREAM the data from a DELTA TABLE and handle the UPDATES AS WELL.**

In [0]:
%sql
CREATE TABLE deltasource
(
  id INT,
  name STRING,
  salary INT 
)
USING DELTA 
LOCATION '/FileStore/deltasource/source1'

In [0]:
%sql
ALTER TABLE deltasource SET TBLPROPERTIES ('delta.enableDeletionVectors' = false)

In [0]:
%sql
INSERT INTO deltasource
VALUES 
(1, "priya", 100),
(2, "rahul", 150),
(3, "shanti", 200)

In [0]:
df = spark.readStream.table("deltasource")

In [0]:
df.writeStream.format("delta")\
    .option("checkpointLocation", "/FileStore/deltasource/sink1/checkpoint")\
    .option("path", "/FileStore/deltasource/sink1/data")\
    .trigger(processingTime = "3 seconds")\
    .start()

in down raw data will be there check there 
It reads data until reserver version 2


In [0]:
%sql
describe history deltasource

It will store reservoir version number in json

In [0]:
#Again insert same data and check
%sql
INSERT INTO deltasource
VALUES 
(1, "priya", 100),
(2, "rahul", 150),
(3, "shanti", 200)


In [0]:
%sql
describe history deltasource

Why 4 ,why not 3

Here we see 3 versions but we are seeing 4 number because ….it says when new records inserts it pull the data from reserver version 4

Now stop the write stream and add 2 times data

In [0]:
%sql
INSERT INTO deltasource
VALUES 
(1, "priya", 100),
(2, "rahul", 150),
(3, "shanti", 200)


In [0]:
%sql
describe history deltasource

When start query ,it automatically read data from version 4---future version to read 

After run it reads 4,5 and showing future version 6

In [0]:
%sql

SELECT * FROM delta.`/FileStore/deltasource/sink1/data`


Insert 3 times again

In [0]:
%sql
INSERT INTO deltasource
VALUES 
(1, "priya", 100),
(2, "rahul", 150),
(3, "shanti", 200)

### **Now new logic **

You don’t want ro read version 6,7..want to read from particular version

In [0]:
%sql
describe history deltasource

Created new one—df_new…because old one not works source and target already link …when you change only source query it not works that why created new source and target

In [0]:
df_new = spark.readStream.option("startingVersion",8).option("ignoreChanges",True).table("deltasource")

df_new.writeStream.format("delta")\
        .option("checkpointLocation","/FileStore/deltasource/sink1/checkpoint_new")\
        .option("path","/FileStore/deltasource/sink1/data_new")\
        .trigger(processingTime = "3 seconds")\
        .start()

In [0]:
%sql
DESCRIBE HISTORY deltasource

In [0]:
%sql
SELECT * FROM delta.`/FileStore/deltasource/sink1/data_new`

Taken only one version so we see 3 records

### **New logic—update **

In [0]:
%sql

UPDATE deltasource
SET name = "iron" where id = 1;

Read strem stops because it will not accepts changes

start the read stream

It will inset 3 records and reads all partions but did not make any changes

In [0]:
%sql

UPDATE deltasource
SET name = "iron" where id = 2;

It will break query

note-After every 10 files it will create one checkpoint like 10.check point 
Next time onwards it didn’t read data from starting ..it reads from 10 th 
